In [14]:
# Import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV, KFold, GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

class NESTED_CV:
    def _init(self, datafile):  # <-- fixed: __init_ not init
        # Load dataset
        self.data = pd.read_excel(datafile)

        # Define target and group columns
        self.target_col = "Release"
        self.group_col = "DP_Group"

        # Drop NA rows just in case
        self.data = self.data.dropna(subset=[self.target_col, self.group_col])

        # Remove non-feature columns
        self.feature_cols = [c for c in self.data.columns 
                             if c not in [self.target_col, self.group_col, 
                                          "Experimental_index", "Time", 
                                          "T=0.25", "T=0.5", "T=1.0"]]

        # Store results
        self.results = []

        # Define models and parameter grids
        self.models = {
            "RF": (RandomForestRegressor(), {
                "n_estimators": [50, 100],
                "max_depth": [None, 5, 10]
            }),
            "SVR": (SVR(), {
                "C": [0.1, 1, 10],
                "kernel": ["linear", "rbf"]
            }),
            "LGBM": (LGBMRegressor(), {
                "n_estimators": [50, 100],
                "learning_rate": [0.01, 0.1]
            }),
            "XGB": (XGBRegressor(objective="reg:squarederror"), {
                "n_estimators": [50, 100],
                "max_depth": [3, 5]
            }),
            "AdaBoost": (AdaBoostRegressor(), {
                "n_estimators": [50, 100],
                "learning_rate": [0.01, 0.1]
            }),
            "GradientBoosting": (GradientBoostingRegressor(), {
                "n_estimators": [50, 100],
                "max_depth": [3, 5]
            }),
            "Ridge": (Ridge(), {
                "alpha": [0.1, 1, 10]
            }),
            "Lasso": (Lasso(), {
                "alpha": [0.01, 0.1, 1]
            }),
            "ElasticNet": (ElasticNet(), {
                "alpha": [0.01, 0.1, 1],
                "l1_ratio": [0.1, 0.5, 0.9]
            })
        }

    def run_nested_cv(self, n_splits=5):
        X = self.data[self.feature_cols]
        y = self.data[self.target_col]
        groups = self.data[self.group_col]

        outer_cv = GroupKFold(n_splits=n_splits)

        best_overall = None
        best_mae = float("inf")

        # Loop through all models
        for name, (model, param_grid) in self.models.items():
            inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
            grid_search = GridSearchCV(model, param_grid, cv=inner_cv, scoring="neg_mean_absolute_error")

            mae_scores = []

            # Outer loop
            for train_idx, test_idx in outer_cv.split(X, y, groups):
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

                grid_search.fit(X_train, y_train)
                y_pred = grid_search.predict(X_test)

                mae = mean_absolute_error(y_test, y_pred)
                mae_scores.append(mae)

            mean_mae = np.mean(mae_scores)
            self.results.append((name, mean_mae))

            print(f"Model {name} finished with MAE = {mean_mae:.4f}")

            # Track best model
            if mean_mae < best_mae:
                best_mae = mean_mae
                best_overall = name

        print(f"\n✅ The best model is {best_overall} with MAE = {best_mae:.4f}")

In [16]:
cv_runner = NESTED_CV_("C:/chem papers/Dataset_17_feat.xlsx")

cv_runner.run_nested_cv(n_splits=5)

NameError: name 'NESTED_CV_' is not defined